# 07 - Test Regime Detection + Dynamic 75% Evaluation

Bu notebook iki işi birlikte yapar:

1. **ETTh1 test pencereleri için STL-based regime label üretir**
2. **Dynamic regime-aware 75% keep maskesini test setinde değerlendirir**

Neden gerekli?

Static pruning için test sonucu vardı. Dynamic selection için ise validation regime label'ları vardı ama test label'ları yoktu.  
Dynamic yöntemi testte çalıştırmak için her test penceresinin `trend / seasonal / residual` regime etiketini bilmemiz gerekir.

Bu notebook:

```text
test window -> STL decomposition -> regime label
test window + regime label -> regime-specific dynamic 75% mask
```

akışını uygular.


## 1. Drive bağla ve importlar

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys
import os
import shutil
import random
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from tqdm.auto import tqdm
from statsmodels.tsa.seasonal import STL

## 2. Proje yolları

In [3]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/BIL401_Regime_Head_Pruning"
)

REGIME_DIR = PROJECT_DIR / "regime_detection"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
HEAD_IMPORTANCE_DIR = PROJECT_DIR / "head_importance"
PRUNING_DIR = PROJECT_DIR / "pruning_experiments"

STATIC_DIR = PRUNING_DIR / "b4_static_pruning_25"
DYNAMIC_75_DIR = PRUNING_DIR / "b4_dynamic_regime_aware_75_keep"
DYNAMIC_75_TEST_DIR = PRUNING_DIR / "b4_dynamic_regime_aware_75_keep_test"

REGIME_DIR.mkdir(parents=True, exist_ok=True)
DYNAMIC_75_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("REGIME_DIR:", REGIME_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("HEAD_IMPORTANCE_DIR:", HEAD_IMPORTANCE_DIR)
print("PRUNING_DIR:", PRUNING_DIR)
print("STATIC_DIR:", STATIC_DIR)
print("DYNAMIC_75_DIR:", DYNAMIC_75_DIR)
print("DYNAMIC_75_TEST_DIR:", DYNAMIC_75_TEST_DIR)

PROJECT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning
REGIME_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/regime_detection
CHECKPOINT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints
HEAD_IMPORTANCE_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/head_importance
PRUNING_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments
STATIC_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_static_pruning_25
DYNAMIC_75_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_dynamic_regime_aware_75_keep
DYNAMIC_75_TEST_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_dynamic_regime_aware_75_keep_test


## 3. Time-Series-Library ve dependency hazırlığı

In [4]:
TSLIB_DIR = Path("/content/Time-Series-Library")

if not TSLIB_DIR.exists():
    %cd /content
    !git clone https://github.com/thuml/Time-Series-Library.git
else:
    print("Time-Series-Library already exists:", TSLIB_DIR)

sys.path.insert(0, str(TSLIB_DIR))
print("Python path[0]:", sys.path[0])

/content
Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 2295, done.
remote: Total 2295 (delta 0), reused 0 (delta 0), pack-reused 2295 (from 1)
Receiving objects: 100% (2295/2295), 78.43 MiB | 17.97 MiB/s, done.
Resolving deltas: 100% (1571/1571), done.
Python path[0]: /content/Time-Series-Library


In [5]:
# TSLib importları için minimal paketler
!pip install -q patool sktime scikit-base --no-deps

# Bazı Time-Series-Library model importları için gerekli olabiliyor.
!pip install -q reformer-pytorch --no-deps
!pip install -q local-attention --no-deps
!pip install -q hyper_connections --no-deps
!pip install -q axial_positional_embedding --no-deps
!pip install -q product_key_memory --no-deps
!pip install -q colt5_attention --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 17.5 MB/s eta 0:00:00


In [6]:
drive_data_path = PROJECT_DIR / "data" / "ETTh1.csv"

tslib_data_path = (
    TSLIB_DIR
    / "dataset/ETDataset/ETT-small/ETTh1.csv"
)

tslib_data_path.parent.mkdir(parents=True, exist_ok=True)

if drive_data_path.exists():
    shutil.copy2(drive_data_path, tslib_data_path)
    print("ETTh1 copied from Drive.")
else:
    print("Drive data not found. Downloading ETTh1...")
    !wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv -O /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv

print("Dataset exists:", tslib_data_path.exists())
print("Dataset path:", tslib_data_path)

df = pd.read_csv(tslib_data_path)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Drive data not found. Downloading ETTh1...
Dataset exists: True
Dataset path: /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv
Dataset shape: (17420, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


## 4. Test seti için STL regime label üret

Time-Series-Library ETTh1 split mantığı:

```text
train: 12 * 30 * 24
val:    4 * 30 * 24
test:   4 * 30 * 24
```

`seq_len=336`, `pred_len=96` olduğu için test segmenti `border1 = train + val - seq_len` noktasından başlar.

Böylece test loader ile aynı sayıda pencere elde edilir.


In [7]:
SEQ_LEN = 336
PRED_LEN = 96
STL_PERIOD = 24
TARGET_COL = "OT"

TRAIN_SIZE = 12 * 30 * 24
VAL_SIZE = 4 * 30 * 24
TEST_SIZE = 4 * 30 * 24

test_border1 = TRAIN_SIZE + VAL_SIZE - SEQ_LEN
test_border2 = TRAIN_SIZE + VAL_SIZE + TEST_SIZE

print("TRAIN_SIZE:", TRAIN_SIZE)
print("VAL_SIZE:", VAL_SIZE)
print("TEST_SIZE:", TEST_SIZE)
print("test_border1:", test_border1)
print("test_border2:", test_border2)

test_segment = df.iloc[test_border1:test_border2].reset_index(drop=True)

print("Test segment shape:", test_segment.shape)
display(test_segment.head())
display(test_segment.tail())

num_test_windows = len(test_segment) - SEQ_LEN - PRED_LEN + 1
print("Num test windows:", num_test_windows)

TRAIN_SIZE: 8640
VAL_SIZE: 2880
TEST_SIZE: 2880
test_border1: 11184
test_border2: 14400
Test segment shape: (3216, 8)


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2017-10-10 00:00:00,10.717,4.287,9.417,2.310,1.858,0.944,14.280
1,2017-10-10 01:00:00,11.855,4.555,10.056,2.772,1.888,1.310,14.843
2,2017-10-10 02:00:00,11.788,4.019,9.417,2.452,1.675,0.914,13.999
3,2017-10-10 03:00:00,11.454,3.952,9.452,2.132,1.736,0.944,14.632
4,2017-10-10 04:00:00,12.726,4.220,9.808,2.452,2.224,1.127,15.265


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
3211,2018-02-20 19:00:00,17.549000,2.813,12.189,1.137,5.361,0.579,0.000
3212,2018-02-20 20:00:00,18.285999,3.081,12.793,1.315,5.544,0.609,0.000
3213,2018-02-20 21:00:00,17.415001,2.746,12.260,1.244,5.239,0.609,1.899
3214,2018-02-20 22:00:00,15.807000,2.478,11.122,1.066,4.691,0.548,2.181
3215,2018-02-20 23:00:00,13.932000,2.210,9.879,0.995,3.990,0.518,2.321


Num test windows: 2785


In [8]:
def safe_variance(values):
    values = np.asarray(values, dtype=np.float64)
    if len(values) == 0:
        return 0.0
    return float(np.var(values))


def label_window_with_stl(
    series_window,
    period=24,
):
    stl = STL(
        series_window,
        period=period,
        robust=True,
    )

    result = stl.fit()

    trend = result.trend
    seasonal = result.seasonal
    residual = result.resid

    trend_var = safe_variance(trend)
    seasonal_var = safe_variance(seasonal)
    residual_var = safe_variance(residual)

    total_var = trend_var + seasonal_var + residual_var

    if total_var <= 1e-12:
        trend_score = 0.0
        seasonal_score = 0.0
        residual_score = 0.0
    else:
        trend_score = trend_var / total_var
        seasonal_score = seasonal_var / total_var
        residual_score = residual_var / total_var

    scores = {
        "trend": trend_score,
        "seasonal": seasonal_score,
        "residual": residual_score,
    }

    regime = max(scores, key=scores.get)

    sorted_scores = sorted(
        scores.values(),
        reverse=True,
    )

    top_score = sorted_scores[0]
    second_score = sorted_scores[1]
    confidence_margin = top_score - second_score

    return {
        "regime": regime,
        "trend_var": trend_var,
        "seasonal_var": seasonal_var,
        "residual_var": residual_var,
        "trend_score": trend_score,
        "seasonal_score": seasonal_score,
        "residual_score": residual_score,
        "top_score": top_score,
        "second_score": second_score,
        "confidence_margin": confidence_margin,
    }


In [9]:
test_regime_records = []

target_values = test_segment[TARGET_COL].values.astype(np.float64)

for window_id in tqdm(range(num_test_windows), desc="STL test regime labeling"):
    input_start = window_id
    input_end = window_id + SEQ_LEN
    pred_start = input_end
    pred_end = pred_start + PRED_LEN

    series_window = target_values[input_start:input_end]

    label_info = label_window_with_stl(
        series_window,
        period=STL_PERIOD,
    )

    test_regime_records.append({
        "window_id": window_id,
        "input_start": input_start,
        "input_end": input_end,
        "pred_start": pred_start,
        "pred_end": pred_end,
        **label_info,
    })

test_regime_df = pd.DataFrame(test_regime_records)
test_regime_df["is_confident"] = test_regime_df["confidence_margin"] >= 0.05

print("Test regime df shape:", test_regime_df.shape)
display(test_regime_df.head())
display(test_regime_df["regime"].value_counts())
display(test_regime_df["regime"].value_counts(normalize=True) * 100)
display(test_regime_df.groupby("regime")["confidence_margin"].describe())

STL test regime labeling:   0%|          | 0/2785 [00:00<?, ?it/s]

Test regime df shape: (2785, 16)


,window_id,input_start,input_end,pred_start,pred_end,regime,trend_var,seasonal_var,residual_var,trend_score,seasonal_score,residual_score,top_score,second_score,confidence_margin,is_confident
0,0,0,336,336,432,trend,3.367733,0.669841,1.424668,0.616548,0.122631,0.260821,0.616548,0.260821,0.355727,True
1,1,1,337,337,433,trend,3.310495,0.660139,1.417605,0.614393,0.122515,0.263092,0.614393,0.263092,0.351300,True
2,2,2,338,338,434,trend,3.112839,0.722205,1.416144,0.592788,0.137532,0.269681,0.592788,0.269681,0.323107,True
3,3,3,339,339,435,trend,2.880675,0.763160,1.432564,0.567464,0.150335,0.282201,0.567464,0.282201,0.285264,True
4,4,4,340,340,436,trend,2.782428,0.823377,1.421409,0.553473,0.163784,0.282743,0.553473,0.282743,0.270730,True


,count
regime,
trend,2606
seasonal,125
residual,54


,proportion
regime,
trend,93.572711
seasonal,4.488330
residual,1.938959


,count,mean,std,min,25%,50%,75%,max
regime,,,,,,,,
residual,54.0,0.031678,0.022408,0.000447,0.013884,0.028083,0.044271,0.085091
seasonal,125.0,0.089549,0.065723,0.000060,0.027294,0.067076,0.144704,0.202803
trend,2606.0,0.441959,0.231719,0.000279,0.294280,0.410464,0.649712,0.867921


In [10]:
test_regime_path = REGIME_DIR / "etth1_test_regimes_ot_seq336.csv"
test_confident_path = REGIME_DIR / "etth1_test_regimes_confident_seq336.csv"
test_summary_path = REGIME_DIR / "etth1_test_regime_summary.csv"

test_regime_df.to_csv(test_regime_path, index=False)
test_regime_df[test_regime_df["is_confident"]].to_csv(test_confident_path, index=False)

test_summary = (
    test_regime_df
    .groupby("regime")
    .agg(
        count=("window_id", "count"),
        mean_confidence_margin=("confidence_margin", "mean"),
        median_confidence_margin=("confidence_margin", "median"),
        confident_count=("is_confident", "sum"),
    )
    .reset_index()
)

test_summary["percentage"] = (
    test_summary["count"]
    / len(test_regime_df)
    * 100
)

test_summary["confident_percentage_within_regime"] = (
    test_summary["confident_count"]
    / test_summary["count"]
    * 100
)

test_summary.to_csv(test_summary_path, index=False)

print("Saved test regime labels:")
print(test_regime_path)
print(test_confident_path)
print(test_summary_path)

display(test_summary)

Saved test regime labels:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/regime_detection/etth1_test_regimes_ot_seq336.csv
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/regime_detection/etth1_test_regimes_confident_seq336.csv
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/regime_detection/etth1_test_regime_summary.csv


,regime,count,mean_confidence_margin,median_confidence_margin,confident_count,percentage,confident_percentage_within_regime
0,residual,54,0.031678,0.028083,13,1.938959,24.074074
1,seasonal,125,0.089549,0.067076,74,4.488330,59.200000
2,trend,2606,0.441959,0.410464,2513,93.572711,96.431312


## 5. Dynamic 75% keep head listelerini oluştur

In [11]:
importance_path = HEAD_IMPORTANCE_DIR / "b4_head_importance_all_windows.csv"
importance_df = pd.read_csv(importance_path)

print("Importance df shape:", importance_df.shape)
display(importance_df.head())

assert len(importance_df) == 24, "B4 için 24 head bekleniyor."


def get_top_heads_for_regime(
    importance_df,
    regime,
    keep_ratio=0.75,
):
    importance_col = f"{regime}_importance"

    num_total = len(importance_df)
    num_keep = int(num_total * keep_ratio)

    top_heads = (
        importance_df
        .sort_values(importance_col, ascending=False)
        .head(num_keep)
        .copy()
        .reset_index(drop=True)
    )

    return top_heads


KEEP_RATIO = 0.75

dynamic_keep_dfs = {
    "trend": get_top_heads_for_regime(
        importance_df,
        "trend",
        keep_ratio=KEEP_RATIO,
    ),
    "seasonal": get_top_heads_for_regime(
        importance_df,
        "seasonal",
        keep_ratio=KEEP_RATIO,
    ),
    "residual": get_top_heads_for_regime(
        importance_df,
        "residual",
        keep_ratio=KEEP_RATIO,
    ),
}

for regime, keep_df in dynamic_keep_dfs.items():
    print(f"\n{regime} keep heads:", len(keep_df))
    display(keep_df[["layer", "head", f"{regime}_importance"]])
    assert len(keep_df) == 18, f"{regime} için 18 head açık bekleniyor."

    keep_df.to_csv(
        DYNAMIC_75_TEST_DIR / f"{regime}_dynamic_keep_75_heads.csv",
        index=False,
    )


Importance df shape: (24, 14)


,layer,head,baseline_overall_mse,masked_overall_mse,overall_importance,baseline_trend_mse,masked_trend_mse,trend_importance,baseline_seasonal_mse,masked_seasonal_mse,seasonal_importance,baseline_residual_mse,masked_residual_mse,residual_importance
0,0,0,0.678066,0.682159,0.004092,0.67695,0.680246,0.003296,0.706689,0.708879,0.002190,0.661419,0.671793,0.010374
1,0,1,0.678066,0.695225,0.017158,0.67695,0.697562,0.020611,0.706689,0.695852,-0.010837,0.661419,0.680825,0.019406
2,0,2,0.678066,0.677773,-0.000293,0.67695,0.673978,-0.002973,0.706689,0.685340,-0.021349,0.661419,0.694178,0.032759
3,0,3,0.678066,0.672993,-0.005073,0.67695,0.679229,0.002279,0.706689,0.632469,-0.074220,0.661419,0.668884,0.007465
4,0,4,0.678066,0.687903,0.009836,0.67695,0.699850,0.022900,0.706689,0.646941,-0.059748,0.661419,0.650203,-0.011217



trend keep heads: 18


,layer,head,trend_importance
0,0,4,0.022900
1,0,1,0.020611
2,2,2,0.012905
3,2,6,0.011054
4,1,6,0.010993
5,1,0,0.008527
6,0,7,0.005714
7,1,4,0.005648
8,1,5,0.003802
9,0,0,0.003296



seasonal keep heads: 18


,layer,head,seasonal_importance
0,2,1,0.036730
1,2,3,0.022212
2,2,7,0.009473
3,1,0,0.006389
4,0,0,0.002190
5,2,2,-0.005710
6,0,6,-0.009039
7,0,1,-0.010837
8,0,2,-0.021349
9,0,7,-0.023027



residual keep heads: 18


,layer,head,residual_importance
0,1,3,0.087188
1,1,6,0.056504
2,2,7,0.055780
3,2,1,0.046922
4,1,1,0.042267
5,2,3,0.040165
6,0,5,0.039628
7,1,0,0.039080
8,0,7,0.035285
9,0,6,0.033360


## 6. Static pruning maskesini oku

In [12]:
static_prune_path = (
    HEAD_IMPORTANCE_DIR
    / "summaries"
    / "static_prune_25_percent_heads.csv"
)

static_prune_df = pd.read_csv(static_prune_path)

print("Static prune heads:")
display(static_prune_df[["layer", "head", "overall_importance"]])

assert len(static_prune_df) == 6, "Static 25% pruning için 6 head bekleniyor."


Static prune heads:


,layer,head,overall_importance
0,0,5,-0.011536
1,1,7,-0.010845
2,2,0,-0.009892
3,1,4,-0.008351
4,1,1,-0.005822
5,0,3,-0.005073


## 7. B4 checkpoint yolunu bul

In [13]:
b4_checkpoint_candidates = list(
    CHECKPOINT_DIR.glob(
        "B4_patchtst_etth1_336_dm128_h8/**/checkpoint.pth"
    )
)

print("Found checkpoint candidates:")
for path in b4_checkpoint_candidates:
    print(path)

if len(b4_checkpoint_candidates) == 0:
    raise FileNotFoundError(
        "B4 checkpoint bulunamadı. CHECKPOINT_DIR içini kontrol et."
    )

b4_checkpoint_path = b4_checkpoint_candidates[0]
print("\nSelected checkpoint:")
print(b4_checkpoint_path)

Found checkpoint candidates:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth

Selected checkpoint:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth


## 8. B4 model argümanları

In [14]:
from argparse import Namespace

args = Namespace(
    # task
    task_name="long_term_forecast",
    is_training=0,
    model_id="ETTh1_336_96_dm128_h8",
    model="PatchTST",

    # data
    data="ETTh1",
    root_path="./dataset/ETDataset/ETT-small/",
    data_path="ETTh1.csv",
    features="M",
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",

    # forecasting
    seq_len=336,
    label_len=48,
    pred_len=96,
    seasonal_patterns="Monthly",
    inverse=False,

    # model
    enc_in=7,
    dec_in=7,
    c_out=7,
    d_model=128,
    n_heads=8,
    e_layers=3,
    d_layers=1,
    d_ff=256,
    moving_avg=25,
    factor=3,
    distil=True,
    dropout=0.1,
    embed="timeF",
    activation="gelu",
    output_attention=False,

    # PatchTST related
    patch_len=16,
    stride=8,
    padding_patch="end",
    revin=1,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,

    # optimization / loader
    num_workers=0,
    itr=1,
    train_epochs=10,
    batch_size=32,
    patience=3,
    learning_rate=0.0001,
    des="baseline_b4",
    loss="MSE",
    lradj="type1",
    use_amp=False,

    # GPU
    use_gpu=torch.cuda.is_available(),
    gpu_type="cuda",
    gpu=0,
    use_multi_gpu=False,
    devices="0",

    # other model families, required by some imports
    expand=2,
    d_conv=4,
    top_k=5,
    num_kernels=6,
    channel_independence=0,
    decomp_method="moving_avg",
    use_norm=1,
    down_sampling_layers=0,
    down_sampling_window=1,
    down_sampling_method=None,
    seg_len=48,

    # MLP projection args sometimes expected
    p_hidden_dims=[128, 128],
    p_hidden_layers=2,
)

print(args)

Namespace(task_name='long_term_forecast', is_training=0, model_id='ETTh1_336_96_dm128_h8', model='PatchTST', data='ETTh1', root_path='./dataset/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=336, label_len=48, pred_len=96, seasonal_patterns='Monthly', inverse=False, enc_in=7, dec_in=7, c_out=7, d_model=128, n_heads=8, e_layers=3, d_layers=1, d_ff=256, moving_avg=25, factor=3, distil=True, dropout=0.1, embed='timeF', activation='gelu', output_attention=False, patch_len=16, stride=8, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, num_workers=0, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='baseline_b4', loss='MSE', lradj='type1', use_amp=False, use_gpu=True, gpu_type='cuda', gpu=0, use_multi_gpu=False, devices='0', expand=2, d_conv=4, top_k=5, num_kernels=6, channel_independence=0, decomp_method='moving_avg', use_norm=1, down_s

## 9. Modeli yükle

In [15]:
%cd /content/Time-Series-Library

/content/Time-Series-Library


In [16]:
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

exp = Exp_Long_Term_Forecast(args)
model = exp.model.to(device)

checkpoint = torch.load(
    b4_checkpoint_path,
    map_location=device,
)

model.load_state_dict(checkpoint)
model.eval()

print("Model loaded successfully.")

Device: cuda:0
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
Model loaded successfully.


## 10. Test loader

In [17]:
from data_provider.data_factory import data_provider

test_data, test_loader = data_provider(
    args,
    flag="test",
)

print("Test dataset length:", len(test_data))
print("Test loader batches:", len(test_loader))
print("Test regime labels:", len(test_regime_df))

assert len(test_data) == len(test_regime_df), (
    len(test_data),
    len(test_regime_df),
)

print("Test windows and regime labels match.")

test 2785
Test dataset length: 2785
Test loader batches: 88
Test regime labels: 2785
Test windows and regime labels match.


## 11. DynamicHeadMaskController

In [18]:
class DynamicHeadMaskController:
    def __init__(self, model, num_channels=7):
        self.model = model
        self.num_channels = num_channels
        self.original_forwards = {}
        self.current_mask = None

    def install(self):
        for layer_idx, encoder_layer in enumerate(self.model.encoder.attn_layers):
            attention_layer = encoder_layer.attention

            if layer_idx in self.original_forwards:
                continue

            original_forward = attention_layer.forward
            self.original_forwards[layer_idx] = original_forward

            def make_masked_forward(layer_idx, attention_layer):
                def masked_forward(
                    queries,
                    keys,
                    values,
                    attn_mask,
                    tau=None,
                    delta=None,
                ):
                    B, L, _ = queries.shape
                    _, S, _ = keys.shape
                    H = attention_layer.n_heads

                    queries_proj = attention_layer.query_projection(queries)
                    keys_proj = attention_layer.key_projection(keys)
                    values_proj = attention_layer.value_projection(values)

                    queries_proj = queries_proj.view(B, L, H, -1)
                    keys_proj = keys_proj.view(B, S, H, -1)
                    values_proj = values_proj.view(B, S, H, -1)

                    out, attn = attention_layer.inner_attention(
                        queries_proj,
                        keys_proj,
                        values_proj,
                        attn_mask,
                        tau=tau,
                        delta=delta,
                    )

                    if self.current_mask is not None:
                        mask = self.current_mask.to(out.device)

                        # Global mask: [num_layers, num_heads]
                        if mask.ndim == 2:
                            layer_mask = mask[layer_idx].view(1, 1, H, 1)

                        # Batch mask: [original_batch, num_layers, num_heads]
                        elif mask.ndim == 3:
                            mask_batch = mask.shape[0]

                            # PatchTST encoder B can be original_batch * channel_count
                            if mask_batch != B:
                                if B % mask_batch != 0:
                                    raise ValueError(
                                        f"Cannot expand mask batch {mask_batch} to encoder batch {B}."
                                    )

                                repeat_factor = B // mask_batch
                                mask = mask.repeat_interleave(
                                    repeat_factor,
                                    dim=0,
                                )

                            layer_mask = mask[:, layer_idx, :].view(B, 1, H, 1)

                        else:
                            raise ValueError(
                                f"Unsupported mask shape: {mask.shape}"
                            )

                        out = out * layer_mask

                    out = out.view(B, L, -1)

                    return attention_layer.out_projection(out), attn

                return masked_forward

            attention_layer.forward = make_masked_forward(
                layer_idx,
                attention_layer,
            )

    def remove(self):
        for layer_idx, original_forward in self.original_forwards.items():
            self.model.encoder.attn_layers[layer_idx].attention.forward = original_forward

        self.original_forwards = {}
        self.current_mask = None

    def set_mask(self, mask):
        self.current_mask = mask.clone().float()

    def build_keep_mask_from_df(self, keep_df):
        num_layers = len(self.model.encoder.attn_layers)
        num_heads = self.model.encoder.attn_layers[0].attention.n_heads

        mask = torch.zeros(
            num_layers,
            num_heads,
            dtype=torch.float32,
        )

        for _, row in keep_df.iterrows():
            layer_idx = int(row["layer"])
            head_idx = int(row["head"])
            mask[layer_idx, head_idx] = 1.0

        return mask

    def build_static_prune_mask(self, prune_df):
        num_layers = len(self.model.encoder.attn_layers)
        num_heads = self.model.encoder.attn_layers[0].attention.n_heads

        mask = torch.ones(
            num_layers,
            num_heads,
            dtype=torch.float32,
        )

        for _, row in prune_df.iterrows():
            layer_idx = int(row["layer"])
            head_idx = int(row["head"])
            mask[layer_idx, head_idx] = 0.0

        return mask


In [19]:
mask_controller = DynamicHeadMaskController(
    model,
    num_channels=args.enc_in,
)

mask_controller.install()

num_layers = len(model.encoder.attn_layers)
num_heads = model.encoder.attn_layers[0].attention.n_heads

print("Layers:", num_layers)
print("Heads per layer:", num_heads)
print("Total heads:", num_layers * num_heads)

baseline_mask = torch.ones(num_layers, num_heads)

static_prune_mask = mask_controller.build_static_prune_mask(static_prune_df)

regime_masks = {
    regime: mask_controller.build_keep_mask_from_df(df_keep)
    for regime, df_keep in dynamic_keep_dfs.items()
}

print("\nStatic 25% mask:")
print(static_prune_mask)
print("Static active heads:", int((static_prune_mask == 1).sum().item()))
print("Static pruned heads:", int((static_prune_mask == 0).sum().item()))

for regime, mask in regime_masks.items():
    print(f"\n{regime} dynamic 75 mask:")
    print(mask)
    print("Active heads:", int((mask == 1).sum().item()))
    print("Pruned heads:", int((mask == 0).sum().item()))

    assert int((mask == 1).sum().item()) == 18
    assert int((mask == 0).sum().item()) == 6


Layers: 3
Heads per layer: 8
Total heads: 24

Static 25% mask:
tensor([[1., 1., 1., 0., 1., 0., 1., 1.],
        [1., 0., 1., 1., 0., 1., 1., 0.],
        [0., 1., 1., 1., 1., 1., 1., 1.]])
Static active heads: 18
Static pruned heads: 6

trend dynamic 75 mask:
tensor([[1., 1., 1., 1., 1., 0., 0., 1.],
        [1., 1., 1., 0., 1., 1., 1., 1.],
        [0., 1., 1., 0., 1., 1., 1., 0.]])
Active heads: 18
Pruned heads: 6

seasonal dynamic 75 mask:
tensor([[1., 1., 1., 0., 0., 0., 1., 1.],
        [1., 0., 1., 1., 0., 1., 1., 1.],
        [0., 1., 1., 1., 1., 1., 1., 1.]])
Active heads: 18
Pruned heads: 6

residual dynamic 75 mask:
tensor([[1., 1., 1., 1., 0., 1., 1., 1.],
        [1., 1., 0., 1., 0., 1., 1., 0.],
        [1., 1., 1., 1., 0., 1., 0., 1.]])
Active heads: 18
Pruned heads: 6


## 12. Batch dynamic mask oluşturma

In [20]:
def build_batch_dynamic_mask(
    window_ids,
    regime_df,
    regime_masks,
):
    batch_masks = []

    for window_id in window_ids:
        regime = regime_df.iloc[int(window_id)]["regime"]
        batch_masks.append(regime_masks[regime])

    batch_mask = torch.stack(batch_masks, dim=0)

    return batch_mask


## 13. Metric fonksiyonları

In [21]:
mse_criterion = nn.MSELoss(reduction="none")
mae_criterion = nn.L1Loss(reduction="none")

def compute_test_with_global_mask(
    model,
    loader,
    test_regime_df,
    mask_controller,
    mask,
    device,
    pred_len=96,
    desc="test global mask",
):
    model.eval()
    mask_controller.set_mask(mask)

    all_records = []
    global_index = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(
                batch_x,
                batch_x_mark,
                batch_y,
                batch_y_mark,
            )

            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            batch_size = batch_x.shape[0]

            for i in range(batch_size):
                window_id = global_index + i
                regime = test_regime_df.iloc[window_id]["regime"]

                all_records.append({
                    "window_id": window_id,
                    "regime": regime,
                    "mse": float(mse_per_sample[i].detach().cpu()),
                    "mae": float(mae_per_sample[i].detach().cpu()),
                })

            global_index += batch_size

    result_df = pd.DataFrame(all_records)

    regime_summary = (
        result_df
        .groupby("regime")
        .agg(
            mse=("mse", "mean"),
            mae=("mae", "mean"),
            count=("window_id", "count"),
        )
        .reset_index()
    )

    overall = {
        "test_mse": float(result_df["mse"].mean()),
        "test_mae": float(result_df["mae"].mean()),
    }

    return {
        "overall": overall,
        "regime_summary": regime_summary,
        "window_losses": result_df,
    }


def compute_test_dynamic(
    model,
    loader,
    test_regime_df,
    regime_masks,
    mask_controller,
    device,
    pred_len=96,
    desc="dynamic test",
):
    model.eval()

    all_records = []
    global_index = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            batch_size = batch_x.shape[0]
            window_ids = list(range(global_index, global_index + batch_size))

            batch_dynamic_mask = build_batch_dynamic_mask(
                window_ids=window_ids,
                regime_df=test_regime_df,
                regime_masks=regime_masks,
            )

            mask_controller.set_mask(batch_dynamic_mask)

            outputs = model(
                batch_x,
                batch_x_mark,
                batch_y,
                batch_y_mark,
            )

            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            for i in range(batch_size):
                window_id = global_index + i
                regime = test_regime_df.iloc[window_id]["regime"]

                all_records.append({
                    "window_id": window_id,
                    "regime": regime,
                    "mse": float(mse_per_sample[i].detach().cpu()),
                    "mae": float(mae_per_sample[i].detach().cpu()),
                })

            global_index += batch_size

    result_df = pd.DataFrame(all_records)

    regime_summary = (
        result_df
        .groupby("regime")
        .agg(
            mse=("mse", "mean"),
            mae=("mae", "mean"),
            count=("window_id", "count"),
        )
        .reset_index()
    )

    overall = {
        "test_mse": float(result_df["mse"].mean()),
        "test_mae": float(result_df["mae"].mean()),
    }

    return {
        "overall": overall,
        "regime_summary": regime_summary,
        "window_losses": result_df,
    }


## 14. Test evaluation: baseline, static 25%, dynamic 75%

In [22]:
baseline_test = compute_test_with_global_mask(
    model=model,
    loader=test_loader,
    test_regime_df=test_regime_df,
    mask_controller=mask_controller,
    mask=baseline_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Baseline test",
)

static_25_test = compute_test_with_global_mask(
    model=model,
    loader=test_loader,
    test_regime_df=test_regime_df,
    mask_controller=mask_controller,
    mask=static_prune_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Static 25% test",
)

dynamic_75_test = compute_test_dynamic(
    model=model,
    loader=test_loader,
    test_regime_df=test_regime_df,
    regime_masks=regime_masks,
    mask_controller=mask_controller,
    device=device,
    pred_len=args.pred_len,
    desc="Dynamic 75% test",
)

print("Baseline test overall:")
print(baseline_test["overall"])

print("\nStatic 25% test overall:")
print(static_25_test["overall"])

print("\nDynamic 75% test overall:")
print(dynamic_75_test["overall"])

print("\nBaseline regime summary:")
display(baseline_test["regime_summary"])

print("\nStatic 25% regime summary:")
display(static_25_test["regime_summary"])

print("\nDynamic 75% regime summary:")
display(dynamic_75_test["regime_summary"])

Baseline test:   0%|          | 0/88 [00:00<?, ?it/s]

Static 25% test:   0%|          | 0/88 [00:00<?, ?it/s]

Dynamic 75% test:   0%|          | 0/88 [00:00<?, ?it/s]

Baseline test overall:
{'test_mse': 0.3725455730083387, 'test_mae': 0.39821428153630434}

Static 25% test overall:
{'test_mse': 0.37831396463208394, 'test_mae': 0.40442411937020195}

Dynamic 75% test overall:
{'test_mse': 0.3793675633157391, 'test_mae': 0.40528710067165175}

Baseline regime summary:


,regime,mse,mae,count
0,residual,0.362111,0.447029,54
1,seasonal,0.341076,0.393135,125
2,trend,0.374271,0.397446,2606



Static 25% regime summary:


,regime,mse,mae,count
0,residual,0.371165,0.453792,54
1,seasonal,0.345894,0.398481,125
2,trend,0.380017,0.403686,2606



Dynamic 75% regime summary:


,regime,mse,mae,count
0,residual,0.356958,0.448172,54
1,seasonal,0.355647,0.407259,125
2,trend,0.380970,0.404304,2606


## 15. Overall test comparison

In [23]:
test_comparison = pd.DataFrame([
    {
        "setting": "B4_no_pruning",
        "method": "none",
        "active_heads_per_sample": 24,
        "pruned_heads_per_sample": 0,
        "test_mse": baseline_test["overall"]["test_mse"],
        "test_mae": baseline_test["overall"]["test_mae"],
    },
    {
        "setting": "B4_static_pruning_25",
        "method": "static_overall_importance",
        "active_heads_per_sample": 18,
        "pruned_heads_per_sample": 6,
        "test_mse": static_25_test["overall"]["test_mse"],
        "test_mae": static_25_test["overall"]["test_mae"],
    },
    {
        "setting": "B4_dynamic_regime_aware_75_keep",
        "method": "dynamic_regime_aware",
        "active_heads_per_sample": 18,
        "pruned_heads_per_sample": 6,
        "test_mse": dynamic_75_test["overall"]["test_mse"],
        "test_mae": dynamic_75_test["overall"]["test_mae"],
    },
])

baseline_mse = test_comparison.loc[
    test_comparison["setting"] == "B4_no_pruning",
    "test_mse",
].iloc[0]

baseline_mae = test_comparison.loc[
    test_comparison["setting"] == "B4_no_pruning",
    "test_mae",
].iloc[0]

test_comparison["delta_test_mse"] = (
    test_comparison["test_mse"] - baseline_mse
)

test_comparison["delta_test_mae"] = (
    test_comparison["test_mae"] - baseline_mae
)

test_comparison["relative_mse_change_percent"] = (
    test_comparison["delta_test_mse"]
    / baseline_mse
    * 100
)

test_comparison["relative_mae_change_percent"] = (
    test_comparison["delta_test_mae"]
    / baseline_mae
    * 100
)

display(test_comparison.sort_values("test_mse"))

,setting,method,active_heads_per_sample,pruned_heads_per_sample,test_mse,test_mae,delta_test_mse,delta_test_mae,relative_mse_change_percent,relative_mae_change_percent
0,B4_no_pruning,none,24,0,0.372546,0.398214,0.000000,0.000000,0.000000,0.000000
1,B4_static_pruning_25,static_overall_importance,18,6,0.378314,0.404424,0.005768,0.006210,1.548372,1.559421
2,B4_dynamic_regime_aware_75_keep,dynamic_regime_aware,18,6,0.379368,0.405287,0.006822,0.007073,1.831183,1.776134


## 16. Regime-level test comparison

In [24]:
baseline_regime = baseline_test["regime_summary"].copy()
baseline_regime["setting"] = "B4_no_pruning"

static_regime = static_25_test["regime_summary"].copy()
static_regime["setting"] = "B4_static_pruning_25"

dynamic_regime = dynamic_75_test["regime_summary"].copy()
dynamic_regime["setting"] = "B4_dynamic_regime_aware_75_keep"

test_regime_comparison = pd.concat(
    [
        baseline_regime,
        static_regime,
        dynamic_regime,
    ],
    ignore_index=True,
)

display(test_regime_comparison)

baseline_regime_ref = baseline_regime[
    ["regime", "mse", "mae"]
].rename(
    columns={
        "mse": "baseline_mse",
        "mae": "baseline_mae",
    }
)

static_regime_delta = static_regime.merge(
    baseline_regime_ref,
    on="regime",
    how="left",
)

static_regime_delta["delta_mse"] = (
    static_regime_delta["mse"]
    - static_regime_delta["baseline_mse"]
)

static_regime_delta["delta_mae"] = (
    static_regime_delta["mae"]
    - static_regime_delta["baseline_mae"]
)

static_regime_delta["relative_mse_change_percent"] = (
    static_regime_delta["delta_mse"]
    / static_regime_delta["baseline_mse"]
    * 100
)

static_regime_delta["relative_mae_change_percent"] = (
    static_regime_delta["delta_mae"]
    / static_regime_delta["baseline_mae"]
    * 100
)


dynamic_regime_delta = dynamic_regime.merge(
    baseline_regime_ref,
    on="regime",
    how="left",
)

dynamic_regime_delta["delta_mse"] = (
    dynamic_regime_delta["mse"]
    - dynamic_regime_delta["baseline_mse"]
)

dynamic_regime_delta["delta_mae"] = (
    dynamic_regime_delta["mae"]
    - dynamic_regime_delta["baseline_mae"]
)

dynamic_regime_delta["relative_mse_change_percent"] = (
    dynamic_regime_delta["delta_mse"]
    / dynamic_regime_delta["baseline_mse"]
    * 100
)

dynamic_regime_delta["relative_mae_change_percent"] = (
    dynamic_regime_delta["delta_mae"]
    / dynamic_regime_delta["baseline_mae"]
    * 100
)

print("Static 25% regime delta:")
display(static_regime_delta)

print("Dynamic 75% regime delta:")
display(dynamic_regime_delta)

,regime,mse,mae,count,setting
0,residual,0.362111,0.447029,54,B4_no_pruning
1,seasonal,0.341076,0.393135,125,B4_no_pruning
2,trend,0.374271,0.397446,2606,B4_no_pruning
3,residual,0.371165,0.453792,54,B4_static_pruning_25
4,seasonal,0.345894,0.398481,125,B4_static_pruning_25
5,trend,0.380017,0.403686,2606,B4_static_pruning_25
6,residual,0.356958,0.448172,54,B4_dynamic_regime_aware_75_keep
7,seasonal,0.355647,0.407259,125,B4_dynamic_regime_aware_75_keep
8,trend,0.380970,0.404304,2606,B4_dynamic_regime_aware_75_keep


Static 25% regime delta:


,regime,mse,mae,count,setting,baseline_mse,baseline_mae,delta_mse,delta_mae,relative_mse_change_percent,relative_mae_change_percent
0,residual,0.371165,0.453792,54,B4_static_pruning_25,0.362111,0.447029,0.009054,0.006763,2.500387,1.512901
1,seasonal,0.345894,0.398481,125,B4_static_pruning_25,0.341076,0.393135,0.004818,0.005346,1.412555,1.359946
2,trend,0.380017,0.403686,2606,B4_static_pruning_25,0.374271,0.397446,0.005746,0.006240,1.535223,1.569970


Dynamic 75% regime delta:


,regime,mse,mae,count,setting,baseline_mse,baseline_mae,delta_mse,delta_mae,relative_mse_change_percent,relative_mae_change_percent
0,residual,0.356958,0.448172,54,B4_dynamic_regime_aware_75_keep,0.362111,0.447029,-0.005153,0.001142,-1.422994,0.255520
1,seasonal,0.355647,0.407259,125,B4_dynamic_regime_aware_75_keep,0.341076,0.393135,0.014571,0.014124,4.272022,3.592552
2,trend,0.380970,0.404304,2606,B4_dynamic_regime_aware_75_keep,0.374271,0.397446,0.006698,0.006858,1.789729,1.725393


## 17. Validation sonuçlarını varsa karşılaştırmaya ekle

In [25]:
validation_combined_path = DYNAMIC_75_DIR / "combined_validation_comparison.csv"

if validation_combined_path.exists():
    validation_combined = pd.read_csv(validation_combined_path)
    print("Validation combined results:")
    display(validation_combined.sort_values("validation_mse"))
else:
    validation_combined = None
    print("Validation combined file not found:", validation_combined_path)


Validation combined results:


,setting,method,active_heads_per_sample,pruned_heads_per_sample,validation_mse,validation_mae,delta_validation_mse,delta_validation_mae,relative_mse_change_percent,relative_mae_change_percent
1,B4_static_pruning_25,static_overall_importance,18,6,0.653658,0.550686,-0.024409,-0.004397,-3.599745,-0.792198
3,B4_dynamic_regime_aware_75_keep,dynamic_regime_aware,18,6,0.659669,0.551188,-0.018398,-0.003895,-2.713252,-0.701762
2,B4_dynamic_regime_aware_50_keep,dynamic_regime_aware,12,12,0.667840,0.555278,-0.010226,0.000195,-1.508114,0.035153
0,B4_no_pruning,none,24,0,0.678066,0.555083,0.000000,0.000000,0.000000,0.000000


## 18. Sonuçları kaydet

In [26]:
test_comparison.to_csv(
    DYNAMIC_75_TEST_DIR / "test_overall_comparison.csv",
    index=False,
)

test_regime_comparison.to_csv(
    DYNAMIC_75_TEST_DIR / "test_regime_comparison.csv",
    index=False,
)

static_regime_delta.to_csv(
    DYNAMIC_75_TEST_DIR / "static_25_test_regime_delta.csv",
    index=False,
)

dynamic_regime_delta.to_csv(
    DYNAMIC_75_TEST_DIR / "dynamic_75_test_regime_delta.csv",
    index=False,
)

baseline_test["window_losses"].to_csv(
    DYNAMIC_75_TEST_DIR / "baseline_test_window_losses.csv",
    index=False,
)

static_25_test["window_losses"].to_csv(
    DYNAMIC_75_TEST_DIR / "static_25_test_window_losses.csv",
    index=False,
)

dynamic_75_test["window_losses"].to_csv(
    DYNAMIC_75_TEST_DIR / "dynamic_75_test_window_losses.csv",
    index=False,
)

test_regime_df.to_csv(
    DYNAMIC_75_TEST_DIR / "test_regime_labels_used.csv",
    index=False,
)

# Maskeleri kaydet
mask_df_static = pd.DataFrame(
    static_prune_mask.numpy(),
    index=[f"layer_{i}" for i in range(static_prune_mask.shape[0])],
    columns=[f"head_{j}" for j in range(static_prune_mask.shape[1])],
)
mask_df_static.to_csv(DYNAMIC_75_TEST_DIR / "static_25_mask.csv")

for regime, mask in regime_masks.items():
    mask_df = pd.DataFrame(
        mask.numpy(),
        index=[f"layer_{i}" for i in range(mask.shape[0])],
        columns=[f"head_{j}" for j in range(mask.shape[1])],
    )
    mask_df.to_csv(
        DYNAMIC_75_TEST_DIR / f"{regime}_dynamic_keep_75_mask.csv"
    )

print("Saved test dynamic experiment outputs to:")
print(DYNAMIC_75_TEST_DIR)

print("\nFiles:")
for path in sorted(DYNAMIC_75_TEST_DIR.iterdir()):
    print(path.name)

Saved test dynamic experiment outputs to:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_dynamic_regime_aware_75_keep_test

Files:
baseline_test_window_losses.csv
dynamic_75_test_regime_delta.csv
dynamic_75_test_window_losses.csv
residual_dynamic_keep_75_heads.csv
residual_dynamic_keep_75_mask.csv
seasonal_dynamic_keep_75_heads.csv
seasonal_dynamic_keep_75_mask.csv
static_25_mask.csv
static_25_test_regime_delta.csv
static_25_test_window_losses.csv
test_overall_comparison.csv
test_regime_comparison.csv
test_regime_labels_used.csv
trend_dynamic_keep_75_heads.csv
trend_dynamic_keep_75_mask.csv


## 19. Kısa yorum üret

In [27]:
dynamic_row = test_comparison[
    test_comparison["setting"] == "B4_dynamic_regime_aware_75_keep"
].iloc[0]

static_row = test_comparison[
    test_comparison["setting"] == "B4_static_pruning_25"
].iloc[0]

print("Test overall:")
display(test_comparison.sort_values("test_mse"))

print("\nDynamic 75% test:")
print(
    f"Dynamic regime-aware 75% keep changed test MSE by "
    f"{dynamic_row['delta_test_mse']:.6f} "
    f"({dynamic_row['relative_mse_change_percent']:.3f}%)."
)

print(
    f"Dynamic regime-aware 75% keep changed test MAE by "
    f"{dynamic_row['delta_test_mae']:.6f} "
    f"({dynamic_row['relative_mae_change_percent']:.3f}%)."
)

print("\nStatic 25% test:")
print(
    f"Static 25% pruning changed test MSE by "
    f"{static_row['delta_test_mse']:.6f} "
    f"({static_row['relative_mse_change_percent']:.3f}%)."
)

print(
    f"Static 25% pruning changed test MAE by "
    f"{static_row['delta_test_mae']:.6f} "
    f"({static_row['relative_mae_change_percent']:.3f}%)."
)

print("\nDynamic 75% regime-level test delta:")
display(
    dynamic_regime_delta[
        [
            "regime",
            "baseline_mse",
            "mse",
            "delta_mse",
            "relative_mse_change_percent",
            "baseline_mae",
            "mae",
            "delta_mae",
            "relative_mae_change_percent",
        ]
    ]
)


Test overall:


,setting,method,active_heads_per_sample,pruned_heads_per_sample,test_mse,test_mae,delta_test_mse,delta_test_mae,relative_mse_change_percent,relative_mae_change_percent
0,B4_no_pruning,none,24,0,0.372546,0.398214,0.000000,0.000000,0.000000,0.000000
1,B4_static_pruning_25,static_overall_importance,18,6,0.378314,0.404424,0.005768,0.006210,1.548372,1.559421
2,B4_dynamic_regime_aware_75_keep,dynamic_regime_aware,18,6,0.379368,0.405287,0.006822,0.007073,1.831183,1.776134



Dynamic 75% test:
Dynamic regime-aware 75% keep changed test MSE by 0.006822 (1.831%).
Dynamic regime-aware 75% keep changed test MAE by 0.007073 (1.776%).

Static 25% test:
Static 25% pruning changed test MSE by 0.005768 (1.548%).
Static 25% pruning changed test MAE by 0.006210 (1.559%).

Dynamic 75% regime-level test delta:


,regime,baseline_mse,mse,delta_mse,relative_mse_change_percent,baseline_mae,mae,delta_mae,relative_mae_change_percent
0,residual,0.362111,0.356958,-0.005153,-1.422994,0.447029,0.448172,0.001142,0.255520
1,seasonal,0.341076,0.355647,0.014571,4.272022,0.393135,0.407259,0.014124,3.592552
2,trend,0.374271,0.380970,0.006698,1.789729,0.397446,0.404304,0.006858,1.725393
